# Notebook 2 · Generating a dataset

Machine learning models learn from **examples**. In a real experiment you would throw thousands of balls and measure how far each one went. Here the simulation from Notebook 1 acts as our "experiment".

Each example (row) records:

| Inputs (features) | Outputs (targets) |
|---|---|
| `v0`: launch speed (m/s) | `range`: horizontal distance travelled (m) |
| `angle_deg`: launch angle (°) | `max_height` (m) |
| `b`: drag parameter (1/m) | `flight_time` (s) |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from projectile import simulate, g  # our code from Notebook 1

## 1. Choose random launch conditions

We pick each input **uniformly at random** within a sensible range. Setting a random *seed* means you get the same "random" numbers every time you run the notebook, so your results are reproducible.

In [ ]:
rng = np.random.default_rng(seed=42)
n_launches = 3000

v0s = rng.uniform(5, 50, n_launches)       # launch speed, m/s
angles = rng.uniform(5, 85, n_launches)    # launch angle, degrees
bs = rng.uniform(0, 0.02, n_launches)      # drag parameter b = k/m, 1/m

## 2. Run the simulations

This runs 3000 simulations and takes roughly **30–60 seconds**. Wait until the `[*]` next to the cell turns into a number.

In [ ]:
rows = []
for v0, angle, b in zip(v0s, angles, bs):
    t, s = simulate(v0, angle, b)
    rows.append({
        "v0": v0,
        "angle_deg": angle,
        "b": b,
        "range": s[-1, 0],
        "max_height": s[:, 1].max(),
        "flight_time": t[-1],
    })

df = pd.DataFrame(rows)
df.head()

In [ ]:
df.describe().round(3)

## 3. Save the dataset

In [ ]:
Path("data").mkdir(exist_ok=True)
df.to_csv("data/projectiles.csv", index=False)
print(f"Saved {len(df)} launches to data/projectiles.csv")

## 4. Explore the data

Always look at your data before doing machine learning.

### How much does drag reduce the range?
The dashed line shows "no air resistance". Each dot is one launch, coloured by its drag parameter $b$.

In [ ]:
no_drag_range = df["v0"]**2 * np.sin(2 * np.radians(df["angle_deg"])) / g

plt.figure(figsize=(7, 6))
plt.scatter(no_drag_range, df["range"], c=df["b"], s=6, cmap="viridis")
plt.colorbar(label="drag parameter b (1/m)")
plt.plot([0, 260], [0, 260], "k--", label="no air resistance")
plt.xlabel("range predicted by SUVAT (m)")
plt.ylabel("actual range with drag (m)")
plt.legend()
plt.show()

Drag matters most for **fast** launches with **large $b$**, because drag force grows with $v^2$.

### The best launch angle is no longer 45°!
Without drag, the best angle for maximum range is $45°$. Does that change once drag is included?

In [ ]:
angles_fine = np.arange(5, 86, 1)

plt.figure(figsize=(9, 5))
for b in [0, 0.005, 0.01, 0.02]:
    ranges = [simulate(30, a, b)[1][-1, 0] for a in angles_fine]
    best = angles_fine[np.argmax(ranges)]
    plt.plot(angles_fine, ranges, label=f"b = {b}  (best angle ≈ {best}°)")
plt.xlabel("launch angle (°)")
plt.ylabel("range (m)")
plt.title("Range vs angle at 30 m/s")
plt.legend()
plt.show()

With air resistance the best angle is **less than 45°**. The projectile spends less time in the air at a lower angle, so drag has less time to slow it down. This would make a good discussion point for your write-up.

**Next:** open `03_machine_learning.ipynb`.